In [1]:
import os
import gc
import numpy as np
import pandas as pd

# 1. Khai báo đường dẫn dữ liệu
raw_prev_path = '/Users/nguyenminhtri/FinalYearPro/data/raw/previous_application.csv'

print(f"📥 Đang nạp file raw previous_application từ: {raw_prev_path}")
df_prev = pd.read_csv(raw_prev_path)

print("=" * 60)
print(f"📊 Kích thước bảng gốc : {df_prev.shape[0]:,} dòng | {df_prev.shape[1]} cột")
print(f"💾 Dung lượng RAM      : {df_prev.memory_usage().sum() / 1024**2:.2f} MB")
print("=" * 60)
df_prev.head()

📥 Đang nạp file raw previous_application từ: /Users/nguyenminhtri/FinalYearPro/data/raw/previous_application.csv
📊 Kích thước bảng gốc : 1,670,214 dòng | 37 cột
💾 Dung lượng RAM      : 671.81 MB


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# --- CELL: DROP NOISY / LOW-IMPORTANCE COLUMNS (VERSION 2) ---

# Danh sách 19 cột nhiễu & metadata cần loại bỏ để tối ưu hiệu suất và giảm overfitting
cols_to_drop = [
    'WEEKDAY_APPR_PROCESS_START',
    'HOUR_APPR_PROCESS_START',
    'FLAG_LAST_APPL_PER_CONTRACT',
    'NFLAG_LAST_APPL_IN_DAY',
    'NFLAG_MICRO_CASH',
    'NAME_CASH_LOAN_PURPOSE',
    'NAME_PAYMENT_TYPE',
    'CODE_REJECT_REASON',
    'NAME_TYPE_SUITE',
    'NAME_CLIENT_TYPE',
    'NAME_GOODS_CATEGORY',
    'NAME_PORTFOLIO',
    'NAME_PRODUCT_TYPE',
    'CHANNEL_TYPE',
    'SELLERPLACE_AREA',
    'NAME_SELLER_INDUSTRY',
    'NAME_YIELD_GROUP',
    'PRODUCT_COMBINATION',
    'NFLAG_INSURED_ON_APPROVAL'
]

print(f"⏳ Đang loại bỏ {len(cols_to_drop)} cột nhiễu khỏi df_prev...")

# Thực hiện xóa an toàn (chỉ xóa các cột thực sự tồn tại)
existing_drops = [col for col in cols_to_drop if col in df_prev.columns]
df_prev.drop(columns=existing_drops, inplace=True, errors='ignore')

print(f"✅ Đã xóa thành công {len(existing_drops)} cột!")
print(f"📊 Kích thước dữ liệu còn lại: {df_prev.shape[0]:,} dòng | {df_prev.shape[1]} cột")

⏳ Đang loại bỏ 19 cột nhiễu khỏi df_prev...
✅ Đã xóa thành công 18 cột!
📊 Kích thước dữ liệu còn lại: 1,670,214 dòng | 19 cột


In [3]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - AMT_ANNUITY ---

# 1. Tạo cờ ghi nhận trạng thái khuyết dữ liệu (Missing Flag)
df_prev['PREV_ANNUITY_IS_NA'] = df_prev['AMT_ANNUITY'].isna().astype(int)

# 2. Xử lý ngoại lai cực đoan bằng Quantile Capping tại bách phân vị 99.9%
annuity_cap = df_prev['AMT_ANNUITY'].quantile(0.999)
df_prev['AMT_ANNUITY_CLEAN'] = df_prev['AMT_ANNUITY'].clip(upper=annuity_cap)

# 3. Điền giá trị khuyết bằng 0.0 cho cột sạch để an toàn khi Groupby / Tích toán
df_prev['AMT_ANNUITY_CLEAN'] = df_prev['AMT_ANNUITY_CLEAN'].fillna(0.0)

# 4. Tạo chỉ số tỷ lệ gánh nặng trả góp trên hạn mức vay (Annuity-to-Credit Ratio)
if 'AMT_CREDIT' in df_prev.columns:
    df_prev['PREV_ANNUITY_TO_CREDIT'] = df_prev['AMT_ANNUITY_CLEAN'] / (df_prev['AMT_CREDIT'] + 1e-5)

print("✅ Đã xử lý xong AMT_ANNUITY!")
print(f"   • Ngưỡng cắt ngoại lai 99.9%: {annuity_cap:,.2f}")
print(f"   • Số lượng bản ghi khuyết được tạo cờ & điền 0.0: {df_prev['PREV_ANNUITY_IS_NA'].sum():,} dòng")
print(f"📊 Kích thước dữ liệu hiện tại: {df_prev.shape[0]:,} dòng | {df_prev.shape[1]} cột")

✅ Đã xử lý xong AMT_ANNUITY!
   • Ngưỡng cắt ngoại lai 99.9%: 119,231.41
   • Số lượng bản ghi khuyết được tạo cờ & điền 0.0: 372,235 dòng
📊 Kích thước dữ liệu hiện tại: 1,670,214 dòng | 22 cột


### 📌 Preprocessing & Feature Engineering: `AMT_ANNUITY` (Monthly Annuity of Previous Application)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_ANNUITY_IS_NA` to preserve missingness signal (22.29% missing rate).
  2. Applied upper-bound quantile capping at the 99.9th percentile to generate `AMT_ANNUITY_CLEAN`, neutralizing extreme outliers.
  3. Imputed missing entries (`NaN`) with `0.0` directly in `AMT_ANNUITY_CLEAN` for numeric aggregation safety.
  4. Derived annuity-to-credit ratio: `PREV_ANNUITY_TO_CREDIT` = `AMT_ANNUITY_CLEAN / (AMT_CREDIT + 1e-5)`.

- **Rationale:**
  - High missing rate (22.29%) corresponds to canceled, refused, or unused loan applications that never finalized a repayment schedule.
  - Imputing `0.0` ensures arithmetic safety when computing total historical monthly annuity obligations per client during downstream `groupby` aggregation.
  - Quantile capping neutralizes extreme outliers while preserving valid high-value loan annuity distributions.

In [4]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - AMT_APPLICATION ---

# 1. Tạo cờ nhị phân ghi nhận đơn xin vay có số tiền yêu cầu bằng 0.0 (Unused / Canceled offer)
df_prev['PREV_APP_IS_ZERO'] = (df_prev['AMT_APPLICATION'] == 0.0).astype(int)

# 2. Xử lý ngoại lai cực đoan bằng Quantile Capping tại bách phân vị 99.9%
app_cap = df_prev['AMT_APPLICATION'].quantile(0.999)
df_prev['AMT_APPLICATION_CLEAN'] = df_prev['AMT_APPLICATION'].clip(upper=app_cap)

# 3. Tỷ lệ giữa số tiền được phê duyệt (AMT_CREDIT) vs số tiền đề xuất xin vay (AMT_APPLICATION_CLEAN)
if 'AMT_CREDIT' in df_prev.columns:
    df_prev['PREV_APP_CREDIT_PERC'] = df_prev['AMT_APPLICATION_CLEAN'] / (df_prev['AMT_CREDIT'] + 1e-5)
    df_prev['PREV_APP_CREDIT_DIFF'] = df_prev['AMT_APPLICATION_CLEAN'] - df_prev['AMT_CREDIT']
    df_prev['PREV_IS_DOWNSIZED'] = (df_prev['PREV_APP_CREDIT_DIFF'] > 0).astype(int)

print("✅ Đã xử lý xong AMT_APPLICATION!")
print(f"   • Ngưỡng gọt mép 99.9%: {app_cap:,.2f}")
print(f"   • Số đơn có số tiền đề xuất = 0.0: {df_prev['PREV_APP_IS_ZERO'].sum():,} đơn")
print(f"📊 Kích thước dữ liệu hiện tại: {df_prev.shape[0]:,} dòng | {df_prev.shape[1]} cột")

✅ Đã xử lý xong AMT_APPLICATION!
   • Ngưỡng gọt mép 99.9%: 2,250,000.00
   • Số đơn có số tiền đề xuất = 0.0: 392,402 đơn
📊 Kích thước dữ liệu hiện tại: 1,670,214 dòng | 27 cột


### 📌 Preprocessing & Feature Engineering: `AMT_APPLICATION` (Requested Credit Amount)

- **Action Taken:**
  1. Constructed binary indicator flag `PREV_APP_IS_ZERO` (`AMT_APPLICATION == 0.0`) to capture zero-amount applications (unused offers / system placeholders).
  2. Applied upper-bound quantile capping at the 99.9th percentile to create `AMT_APPLICATION_CLEAN`, neutralizing extreme right-tail outliers.
  3. Calculated requested-to-granted credit ratio: `PREV_APP_CREDIT_PERC` = `AMT_APPLICATION_CLEAN / (AMT_CREDIT + 1e-5)`.
  4. Derived absolute limit difference: `PREV_APP_CREDIT_DIFF` = `AMT_APPLICATION_CLEAN - AMT_CREDIT`.
  5. Created downsized approval indicator flag: `PREV_IS_DOWNSIZED` (`PREV_APP_CREDIT_DIFF > 0`).

- **Rationale:**
  - `AMT_APPLICATION` has a 0.00% missing rate but exhibits extreme right-skewness (Max = 6.9M vs Median = 71k). Quantile capping preserves high-value loan variance while preventing split distortions in tree models.
  - Comparing requested amount (`AMT_APPLICATION_CLEAN`) against actual approved amount (`AMT_CREDIT`) captures historical lender risk appetite—applications where requested credit exceeds granted credit (`PREV_IS_DOWNSIZED == 1`) signal prior credit risk mitigation by Home Credit.

In [5]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - AMT_CREDIT ---

# 1. Tạo cờ nhị phân ghi nhận đơn xin vay có số tiền phê duyệt bằng 0.0 (Refused / Canceled loans)
df_prev['PREV_CREDIT_IS_ZERO'] = (df_prev['AMT_CREDIT'] == 0.0).astype(int)

# 2. Xử lý ngoại lai cực đoan bằng Quantile Capping tại bách phân vị 99.9%
credit_cap = df_prev['AMT_CREDIT'].quantile(0.999)
df_prev['AMT_CREDIT_CLEAN'] = df_prev['AMT_CREDIT'].clip(upper=credit_cap)

print("✅ Đã xử lý xong AMT_CREDIT!")
print(f"   • Ngưỡng gọt mép 99.9%: {credit_cap:,.2f}")
print(f"   • Số đơn có số tiền phê duyệt = 0.0: {df_prev['PREV_CREDIT_IS_ZERO'].sum():,} đơn")
print(f"📊 Kích thước dữ liệu hiện tại: {df_prev.shape[0]:,} dòng | {df_prev.shape[1]} cột")

✅ Đã xử lý xong AMT_CREDIT!
   • Ngưỡng gọt mép 99.9%: 2,517,300.00
   • Số đơn có số tiền phê duyệt = 0.0: 336,768 đơn
📊 Kích thước dữ liệu hiện tại: 1,670,214 dòng | 29 cột


### 📌 Preprocessing & Feature Engineering: `AMT_CREDIT` (Approved Credit Amount of Previous Application)

- **Action Taken:**
  1. Constructed binary indicator flag `PREV_CREDIT_IS_ZERO` (`AMT_CREDIT == 0.0`) to explicitly identify applications with zero approved credit (refused, canceled, or ungranted offers).
  2. Applied upper-bound quantile capping at the 99.9th percentile to generate `AMT_CREDIT_CLEAN`, neutralizing extreme right-tail outliers.

- **Rationale:**
  - `AMT_CREDIT` exhibits 0.00% missing rate but features severe right-skewness (Max = 6.9M vs Median = 80.5k) and a mode at 0.00 representing non-disbursed applications.
  - Capping extreme tail values at the 99.9th percentile prevents split distortion in tree-based algorithms while maintaining valid variance across approved credit tiers.
  - Isolating zero-credit records (`PREV_CREDIT_IS_ZERO == 1`) creates a clear behavioral marker for unapproved loan attempts prior to aggregation.

In [6]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - AMT_DOWN_PAYMENT ---

# 1. Create binary indicator flag for missing values (Missing Rate: 53.64%)
df_prev['PREV_DOWN_PAYMENT_IS_NA'] = df_prev['AMT_DOWN_PAYMENT'].isna().astype(int)

# 2. Neutralize negative accounting noise (Min = -0.90) and apply 99.9th percentile upper quantile capping
dp_cap = df_prev['AMT_DOWN_PAYMENT'].quantile(0.999)
df_prev['AMT_DOWN_PAYMENT_CLEAN'] = df_prev['AMT_DOWN_PAYMENT'].clip(lower=0.0, upper=dp_cap)

# 3. Impute missing values with 0.0 for arithmetic safety during downstream aggregations
df_prev['AMT_DOWN_PAYMENT_CLEAN'] = df_prev['AMT_DOWN_PAYMENT_CLEAN'].fillna(0.0)

# 4. Binary indicator flag for applications with an active up-front cash down payment
df_prev['PREV_HAS_DOWN_PAYMENT'] = (df_prev['AMT_DOWN_PAYMENT_CLEAN'] > 0.0).astype(int)

# 5. Down-payment-to-credit ratio relative to approved credit limit
if 'AMT_CREDIT' in df_prev.columns:
    df_prev['PREV_DOWN_PAYMENT_TO_CREDIT'] = df_prev['AMT_DOWN_PAYMENT_CLEAN'] / (df_prev['AMT_CREDIT'] + 1e-5)

print("✅ Successfully processed AMT_DOWN_PAYMENT!")
print(f"   • 99.9th percentile capping threshold: {dp_cap:,.2f}")
print(
    f"   • Total missing records flagged: {df_prev['PREV_DOWN_PAYMENT_IS_NA'].sum():,} ({df_prev['PREV_DOWN_PAYMENT_IS_NA'].mean() * 100:.2f}%)")
print(f"   • Applications with active down payment (> 0): {df_prev['PREV_HAS_DOWN_PAYMENT'].sum():,}")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed AMT_DOWN_PAYMENT!
   • 99.9th percentile capping threshold: 208,801.96
   • Total missing records flagged: 895,844 (53.64%)
   • Applications with active down payment (> 0): 404,514
📊 Current dataset shape: 1,670,214 rows | 33 columns


### 📌 Preprocessing & Feature Engineering: `AMT_DOWN_PAYMENT` (Up-front Cash Down Payment)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_DOWN_PAYMENT_IS_NA` to preserve missingness structure (53.64% missing rate).
  2. Clipped lower bound at `0.0` (neutralizing accounting noise like `-0.90`) and applied upper-bound quantile capping at the 99.9th percentile to create `AMT_DOWN_PAYMENT_CLEAN`.
  3. Imputed missing entries (`NaN`) with `0.0` in `AMT_DOWN_PAYMENT_CLEAN` to ensure arithmetic safety during downstream aggregations.
  4. Derived binary equity commitment flag: `PREV_HAS_DOWN_PAYMENT` (`AMT_DOWN_PAYMENT_CLEAN > 0.0`).
  5. Formulated down-payment-to-credit ratio: `PREV_DOWN_PAYMENT_TO_CREDIT` = `AMT_DOWN_PAYMENT_CLEAN / (AMT_CREDIT + 1e-5)`.

- **Rationale:**
  - High missingness (53.64%) is structural: cash loans and revolving credit products do not require down payments, whereas consumer loans for physical goods do.
  - Making an up-front cash down payment demonstrates applicant equity commitment and financial capacity, serving as a strong protective signal against credit default.
  - Imputing `0.0` for missing records allows summing total down payments per applicant during `groupby` aggregation without generating `NaN` values.

In [7]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - AMT_GOODS_PRICE ---

# 1. Create binary indicator flag for missing values (Missing Rate: 23.08%)
df_prev['PREV_GOODS_PRICE_IS_NA'] = df_prev['AMT_GOODS_PRICE'].isna().astype(int)

# 2. Apply upper-bound quantile capping at the 99.9th percentile to generate AMT_GOODS_PRICE_CLEAN
gp_cap = df_prev['AMT_GOODS_PRICE'].quantile(0.999)
df_prev['AMT_GOODS_PRICE_CLEAN'] = df_prev['AMT_GOODS_PRICE'].clip(upper=gp_cap)

# 3. Domain-driven imputation: Fill missing AMT_GOODS_PRICE with AMT_CREDIT (Cash loans baseline)
if 'AMT_CREDIT' in df_prev.columns:
    df_prev['AMT_GOODS_PRICE_CLEAN'] = df_prev['AMT_GOODS_PRICE_CLEAN'].fillna(df_prev['AMT_CREDIT'])
else:
    df_prev['AMT_GOODS_PRICE_CLEAN'] = df_prev['AMT_GOODS_PRICE_CLEAN'].fillna(0.0)

# 4. Formulate loan-to-goods financing ratio (Credit / Goods Price)
if 'AMT_CREDIT' in df_prev.columns:
    df_prev['PREV_GOODS_CREDIT_RATIO'] = df_prev['AMT_CREDIT'] / (df_prev['AMT_GOODS_PRICE_CLEAN'] + 1e-5)

print("✅ Successfully processed AMT_GOODS_PRICE!")
print(f"   • 99.9th percentile capping threshold: {gp_cap:,.2f}")
print(
    f"   • Total missing records flagged and imputed via AMT_CREDIT: {df_prev['PREV_GOODS_PRICE_IS_NA'].sum():,} ({df_prev['PREV_GOODS_PRICE_IS_NA'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed AMT_GOODS_PRICE!
   • 99.9th percentile capping threshold: 2,250,000.00
   • Total missing records flagged and imputed via AMT_CREDIT: 385,515 (23.08%)
📊 Current dataset shape: 1,670,214 rows | 36 columns


### 📌 Preprocessing & Feature Engineering: `AMT_GOODS_PRICE` (Price of Goods in Previous Application)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_GOODS_PRICE_IS_NA` to retain missingness structure (23.08% missing rate).
  2. Applied upper-bound quantile capping at the 99.9th percentile to create `AMT_GOODS_PRICE_CLEAN`, suppressing extreme tail outliers (Max = 6.9M vs Median = 112.3k).
  3. Performed domain-driven imputation: filled missing entries in `AMT_GOODS_PRICE_CLEAN` with `AMT_CREDIT` values.
  4. Formulated loan-to-goods financing ratio: `PREV_GOODS_CREDIT_RATIO` = `AMT_CREDIT / (AMT_GOODS_PRICE_CLEAN + 1e-5)`.

- **Rationale:**
  - High missing rate (23.08%) is structurally tied to cash loans and revolving credit products where no underlying consumer good exists.
  - Imputing missing values with `AMT_CREDIT` aligns with banking domain logic for cash loans, ensuring a baseline financing ratio of `1.0` during downstream aggregations.
  - Quantile capping neutralizes extreme right-tail distortion while preserving high-value merchandise pricing distribution.

In [8]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - RATE_DOWN_PAYMENT ---

# 1. Create binary indicator flag for missing values (Missing Rate: 53.64%)
df_prev['PREV_RATE_DOWN_PAYMENT_IS_NA'] = df_prev['RATE_DOWN_PAYMENT'].isna().astype(int)

# 2. Clip values strictly within the valid percentage range [0.0, 1.0] to eliminate negative rounding artifacts (-0.00)
df_prev['RATE_DOWN_PAYMENT_CLEAN'] = df_prev['RATE_DOWN_PAYMENT'].clip(lower=0.0, upper=1.0)

# 3. Impute missing values with 0.0 for arithmetic safety during downstream aggregations
df_prev['RATE_DOWN_PAYMENT_CLEAN'] = df_prev['RATE_DOWN_PAYMENT_CLEAN'].fillna(0.0)

print("✅ Successfully processed RATE_DOWN_PAYMENT!")
print(
    f"   • Total missing records flagged and imputed (0.0): {df_prev['PREV_RATE_DOWN_PAYMENT_IS_NA'].sum():,} ({df_prev['PREV_RATE_DOWN_PAYMENT_IS_NA'].mean() * 100:.2f}%)")
print(f"   • Clean mean down-payment rate: {df_prev['RATE_DOWN_PAYMENT_CLEAN'].mean():.4f}")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed RATE_DOWN_PAYMENT!
   • Total missing records flagged and imputed (0.0): 895,844 (53.64%)
   • Clean mean down-payment rate: 0.0369
📊 Current dataset shape: 1,670,214 rows | 38 columns


### 📌 Preprocessing & Feature Engineering: `RATE_DOWN_PAYMENT` (Normalized Down Payment Rate)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_RATE_DOWN_PAYMENT_IS_NA` to preserve missingness structure (53.64% missing rate).
  2. Clipped feature values strictly to the $[0.0, 1.0]$ interval to generate `RATE_DOWN_PAYMENT_CLEAN`, neutralizing negative accounting rounding noise (e.g., `-0.00`).
  3. Imputed missing entries (`NaN`) with `0.0` in `RATE_DOWN_PAYMENT_CLEAN` to ensure arithmetic safety during downstream `groupby` aggregations.

- **Rationale:**
  - The 53.64% missing rate aligns structurally with `AMT_DOWN_PAYMENT`, corresponding to non-POS loan products (cash loans and revolving credit) where up-front down payments do not apply.
  - Imputing `0.0` for missing records reflects the zero-percentage down payment nature of cash loans while allowing arithmetic aggregations (`MEAN`, `MAX`) per client without propagating `NaN` values.
  - The distribution features distinct multimodal spikes at standard commercial financing tiers (10%, 20%, 30%), capturing key product-level risk segmentation.

In [9]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - RATE_INTEREST_PRIMARY ---

# 1. Create binary indicator flag for missing values (Extreme Missing Rate: 99.64%)
df_prev['PREV_RATE_INTEREST_PRIMARY_IS_NA'] = df_prev['RATE_INTEREST_PRIMARY'].isna().astype(int)

# 2. Drop the original raw feature due to prohibitive missingness (>99%) to prevent noise and overfitting
df_prev.drop(columns=['RATE_INTEREST_PRIMARY'], inplace=True, errors='ignore')

print("✅ Successfully processed RATE_INTEREST_PRIMARY!")
print(f"   • Extreme missing rate detected: 99.64% (only 5,951 valid records)")
print(f"   • Retained binary indicator flag 'PREV_RATE_INTEREST_PRIMARY_IS_NA'")
print(f"   • Dropped continuous feature to eliminate synthetic noise")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed RATE_INTEREST_PRIMARY!
   • Extreme missing rate detected: 99.64% (only 5,951 valid records)
   • Retained binary indicator flag 'PREV_RATE_INTEREST_PRIMARY_IS_NA'
   • Dropped continuous feature to eliminate synthetic noise
📊 Current dataset shape: 1,670,214 rows | 38 columns


### 📌 Preprocessing & Feature Engineering: `RATE_INTEREST_PRIMARY` (Primary Interest Rate)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_RATE_INTEREST_PRIMARY_IS_NA` to preserve the structural missingness signal (99.64% missing rate).
  2. Dropped the original continuous column `RATE_INTEREST_PRIMARY` to prevent model overfitting on an extremely sparse subset (0.36% non-null).

- **Rationale:**
  - With 99.64% missing data (only 5,951 valid records out of 1.67M), standard imputation methods (mean/median/zero-imputation) would inject massive synthetic noise.
  - Converting the feature into a binary indicator captures potential segment signals (e.g., special promotional interest rate programs applied to a tiny tier of customers) while discarding the noisy continuous variable.

In [10]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - RATE_INTEREST_PRIVILEGED ---

# 1. Create binary indicator flag for missing values (Extreme Missing Rate: 99.64%)
df_prev['PREV_RATE_INTEREST_PRIVILEGED_IS_NA'] = df_prev['RATE_INTEREST_PRIVILEGED'].isna().astype(int)

# 2. Drop the original raw feature due to prohibitive missingness (>99%) to prevent noise and overfitting
df_prev.drop(columns=['RATE_INTEREST_PRIVILEGED'], inplace=True, errors='ignore')

print("✅ Successfully processed RATE_INTEREST_PRIVILEGED!")
print(f"   • Extreme missing rate detected: 99.64% (only 5,951 valid records)")
print(f"   • Retained binary indicator flag 'PREV_RATE_INTEREST_PRIVILEGED_IS_NA'")
print(f"   • Dropped continuous feature to eliminate synthetic noise")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed RATE_INTEREST_PRIVILEGED!
   • Extreme missing rate detected: 99.64% (only 5,951 valid records)
   • Retained binary indicator flag 'PREV_RATE_INTEREST_PRIVILEGED_IS_NA'
   • Dropped continuous feature to eliminate synthetic noise
📊 Current dataset shape: 1,670,214 rows | 38 columns


### 📌 Preprocessing & Feature Engineering: `RATE_INTEREST_PRIVILEGED` (Privileged Interest Rate)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_RATE_INTEREST_PRIVILEGED_IS_NA` to preserve the structural missingness signal (99.64% missing rate).
  2. Dropped the original continuous column `RATE_INTEREST_PRIVILEGED` to prevent model overfitting on an extremely sparse subset (0.36% non-null).

- **Rationale:**
  - With 99.64% missing data (only 5,951 valid records out of 1.67M), applying continuous imputation techniques would introduce severe synthetic noise.
  - Converting the feature into a binary missingness indicator captures potential segment signals (e.g., special promotional/privileged interest rate schemes) while removing sparse continuous variance.

In [11]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - NAME_CONTRACT_STATUS ---

# 1. One-hot encoding binary indicator flags for contract status categories (0.00% missing)
df_prev['PREV_IS_APPROVED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
df_prev['PREV_IS_CANCELED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Canceled').astype(int)
df_prev['PREV_IS_REFUSED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
df_prev['PREV_IS_UNUSED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Unused offer').astype(int)

print("✅ Successfully processed NAME_CONTRACT_STATUS!")
print(
    f"   • Approved applications : {df_prev['PREV_IS_APPROVED'].sum():,} ({df_prev['PREV_IS_APPROVED'].mean() * 100:.2f}%)")
print(
    f"   • Canceled applications : {df_prev['PREV_IS_CANCELED'].sum():,} ({df_prev['PREV_IS_CANCELED'].mean() * 100:.2f}%)")
print(
    f"   • Refused applications  : {df_prev['PREV_IS_REFUSED'].sum():,} ({df_prev['PREV_IS_REFUSED'].mean() * 100:.2f}%)")
print(
    f"   • Unused offer loans    : {df_prev['PREV_IS_UNUSED'].sum():,} ({df_prev['PREV_IS_UNUSED'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed NAME_CONTRACT_STATUS!
   • Approved applications : 1,036,781 (62.07%)
   • Canceled applications : 316,319 (18.94%)
   • Refused applications  : 290,678 (17.40%)
   • Unused offer loans    : 26,436 (1.58%)
📊 Current dataset shape: 1,670,214 rows | 42 columns


### 📌 Preprocessing & Feature Engineering: `NAME_CONTRACT_STATUS` (Contract Approval Status)

- **Action Taken:**
  1. One-hot encoded categorical contract statuses into 4 distinct binary indicator flags: `PREV_IS_APPROVED`, `PREV_IS_CANCELED`, `PREV_IS_REFUSED`, and `PREV_IS_UNUSED` (0.00% missing rate).
  2. Prepared flags for multi-segment aggregation to isolate behavioral risk profiles per applicant.

- **Rationale:**
  - `NAME_CONTRACT_STATUS` contains zero missing values and serves as one of the single most predictive categorical features in `previous_application`.
  - Converting categories into explicit binary flags enables calculating critical behavioral indicators during downstream `groupby` aggregation, such as client-level refusal rates (`PREV_REFUSAL_RATE`) and approval counts.
  - A high historical refusal rate (`Refused = 17.40%`) strongly correlates with default risk on current loan applications.

In [12]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_DECISION ---

# 1. Convert negative relative days to positive elapsed years for intuitive feature interpretation
df_prev['PREV_DECISION_YEARS'] = df_prev['DAYS_DECISION'].abs() / 365.25

# 2. Create recency binary indicator flags for decision timeframes (1-Year & 2-Years windows)
df_prev['PREV_IS_RECENT_1Y'] = (df_prev['DAYS_DECISION'] >= -365).astype(int)
df_prev['PREV_IS_RECENT_2Y'] = (df_prev['DAYS_DECISION'] >= -730).astype(int)

print("✅ Successfully processed DAYS_DECISION!")
print(f"   • Application recency range: {df_prev['DAYS_DECISION'].max()} to {df_prev['DAYS_DECISION'].min()} days")
print(
    f"   • Applications within last 1 year (<= 365 days): {df_prev['PREV_IS_RECENT_1Y'].sum():,} ({df_prev['PREV_IS_RECENT_1Y'].mean() * 100:.2f}%)")
print(
    f"   • Applications within last 2 years (<= 730 days): {df_prev['PREV_IS_RECENT_2Y'].sum():,} ({df_prev['PREV_IS_RECENT_2Y'].mean() * 100:.2f}%)")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_DECISION!
   • Application recency range: -1 to -2922 days
   • Applications within last 1 year (<= 365 days): 573,740 (34.35%)
   • Applications within last 2 years (<= 730 days): 958,838 (57.41%)
📊 Current dataset shape: 1,670,214 rows | 45 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_DECISION` (Application Decision Timeframe)

- **Action Taken:**
  1. Converted raw negative relative decision days into absolute positive years: `PREV_DECISION_YEARS` = `abs(DAYS_DECISION) / 365.25`.
  2. Constructed recency binary indicator flags: `PREV_IS_RECENT_1Y` (`DAYS_DECISION >= -365`) and `PREV_IS_RECENT_2Y` (`DAYS_DECISION >= -730`) to isolate recent borrower behavior.
  3. Retained raw `DAYS_DECISION` to support multi-period min/max aggregations (capturing most recent vs. oldest application dates per applicant).

- **Rationale:**
  - `DAYS_DECISION` has 0.00% missing values and ranges from -1 to -2,922 days (~8 years).
  - Recent credit applications are significantly more predictive of current financial status than historical applications made years ago (time-decay effect).
  - Time-window flags (`1Y`, `2Y`) allow downstream `groupby` aggregations to measure recent application frequency, capturing credit-seeking behavior intensity.

In [13]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - CNT_PAYMENT ---

# 1. Create binary indicator flag for missing values (Missing Rate: 22.29%)
df_prev['PREV_CNT_PAYMENT_IS_NA'] = df_prev['CNT_PAYMENT'].isna().astype(int)

# 2. Impute missing values with 0.0 for arithmetic safety during downstream aggregations
df_prev['CNT_PAYMENT_CLEAN'] = df_prev['CNT_PAYMENT'].fillna(0.0)

# 3. Create short-term credit commitment indicator flag (<= 12 months term)
df_prev['PREV_IS_SHORT_TERM'] = ((df_prev['CNT_PAYMENT_CLEAN'] > 0.0) & (df_prev['CNT_PAYMENT_CLEAN'] <= 12.0)).astype(
    int)

# 4. Formulate total historical contract obligation proxy (Clean Annuity * Term Count)
if 'AMT_ANNUITY_CLEAN' in df_prev.columns:
    df_prev['PREV_TOTAL_CONTRACT_VAL'] = df_prev['AMT_ANNUITY_CLEAN'] * df_prev['CNT_PAYMENT_CLEAN']

print("✅ Successfully processed CNT_PAYMENT!")
print(
    f"   • Total missing records flagged and imputed (0.0): {df_prev['PREV_CNT_PAYMENT_IS_NA'].sum():,} ({df_prev['PREV_CNT_PAYMENT_IS_NA'].mean() * 100:.2f}%)")
print(f"   • Applications with short-term tenure (<= 12M): {df_prev['PREV_IS_SHORT_TERM'].sum():,}")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed CNT_PAYMENT!
   • Total missing records flagged and imputed (0.0): 372,230 (22.29%)
   • Applications with short-term tenure (<= 12M): 721,030
📊 Current dataset shape: 1,670,214 rows | 49 columns


### 📌 Preprocessing & Feature Engineering: `CNT_PAYMENT` (Term / Number of Payments of Previous Application)

- **Action Taken:**
  1. Constructed binary missing indicator flag `PREV_CNT_PAYMENT_IS_NA` to preserve missingness structure (22.29% missing rate).
  2. Imputed missing entries (`NaN`) with `0.0` in `CNT_PAYMENT_CLEAN` for arithmetic aggregation safety.
  3. Derived binary tenure classification flag: `PREV_IS_SHORT_TERM` (`0.0 < CNT_PAYMENT_CLEAN <= 12.0`).
  4. Formulated total estimated contract value proxy: `PREV_TOTAL_CONTRACT_VAL` = `AMT_ANNUITY_CLEAN * CNT_PAYMENT_CLEAN`.

- **Rationale:**
  - Missingness (22.29%) matches `AMT_ANNUITY` exactly, corresponding to unfinalized, canceled, or refused loan applications.
  - Imputing `0.0` ensures safety during `groupby` operations (`MEAN`, `MAX`, `SUM`) without generating `NaN` values for non-disbursed contracts.
  - The distribution shows pronounced multimodal peaks at standard commercial credit terms (6, 12, 18, 24, 36, 60 months). Isolating short-term tenure (`<= 12M`) provides a strong risk-profiling marker for credit scoring models.

In [14]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_FIRST_DRAWING ---

# 1. Replace placeholder value 365243 with NaN to resolve systematic data anomaly
df_prev['DAYS_FIRST_DRAWING'] = df_prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan)

# 2. Create binary indicator flag for missing/un-drawn records
df_prev['PREV_FIRST_DRAWING_IS_NA'] = df_prev['DAYS_FIRST_DRAWING'].isna().astype(int)

# 3. Convert valid negative relative days into positive elapsed years for model interpretability
df_prev['PREV_FIRST_DRAWING_YEARS'] = df_prev['DAYS_FIRST_DRAWING'].abs() / 365.25

print("✅ Successfully processed DAYS_FIRST_DRAWING!")
print(
    f"   • Total missing/un-drawn records flagged: {df_prev['PREV_FIRST_DRAWING_IS_NA'].sum():,} ({df_prev['PREV_FIRST_DRAWING_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Valid draw records range: {df_prev['DAYS_FIRST_DRAWING'].min()} to {df_prev['DAYS_FIRST_DRAWING'].max()} days")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_FIRST_DRAWING!
   • Total missing/un-drawn records flagged: 1,607,509 (96.25%)
   • Valid draw records range: -2922.0 to -2.0 days
📊 Current dataset shape: 1,670,214 rows | 51 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_FIRST_DRAWING` (Relative Days to First Loan Disbursement)

- **Action Taken:**
  1. Replaced systemic anomaly placeholder `365243` (representing ~1,000 years) with `np.nan` to restore physical time semantics.
  2. Constructed binary missing indicator flag `PREV_FIRST_DRAWING_IS_NA` to preserve structural missingness signal (40.30% raw missing + replaced placeholders).
  3. Formulated absolute elapsed time in years for valid disbursement records: `PREV_FIRST_DRAWING_YEARS` = `abs(DAYS_FIRST_DRAWING) / 365.25`.

- **Rationale:**
  - The value `365243` dominates the raw distribution (Median/Mode = 365,243) as a database placeholder for non-disbursed contracts or unused credit lines. Leaving it unhandled distorts decision tree split logic.
  - Converting valid relative negative days to positive years (`PREV_FIRST_DRAWING_YEARS`) provides clean temporal distance for xAI interpretability (SHAP analysis).
  - Preserving the binary flag (`PREV_FIRST_DRAWING_IS_NA`) isolates applications that never reached first cash disbursement.

In [15]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_FIRST_DUE ---

# 1. Replace placeholder value 365243 with NaN to resolve systematic data anomaly
df_prev['DAYS_FIRST_DUE'] = df_prev['DAYS_FIRST_DUE'].replace(365243, np.nan)

# 2. Create binary indicator flag for missing/un-scheduled first due dates
df_prev['PREV_FIRST_DUE_IS_NA'] = df_prev['DAYS_FIRST_DUE'].isna().astype(int)

# 3. Convert valid negative relative days into positive elapsed years for model interpretability
df_prev['PREV_FIRST_DUE_YEARS'] = df_prev['DAYS_FIRST_DUE'].abs() / 365.25

# 4. Calculate day offset between first disbursement and first payment due date
if 'DAYS_FIRST_DRAWING' in df_prev.columns:
    df_prev['PREV_DAYS_DRAWING_TO_DUE'] = df_prev['DAYS_FIRST_DUE'] - df_prev['DAYS_FIRST_DRAWING']

print("✅ Successfully processed DAYS_FIRST_DUE!")
print(
    f"   • Total missing/un-scheduled records flagged: {df_prev['PREV_FIRST_DUE_IS_NA'].sum():,} ({df_prev['PREV_FIRST_DUE_IS_NA'].mean() * 100:.2f}%)")
print(f"   • Valid due records range: {df_prev['DAYS_FIRST_DUE'].min()} to {df_prev['DAYS_FIRST_DUE'].max()} days")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_FIRST_DUE!
   • Total missing/un-scheduled records flagged: 713,710 (42.73%)
   • Valid due records range: -2892.0 to -2.0 days
📊 Current dataset shape: 1,670,214 rows | 54 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_FIRST_DUE` (Relative Days to First Payment Due Date)

- **Action Taken:**
  1. Replaced systemic anomaly placeholder `365243` (representing ~1,000 years in the future) with `np.nan` to restore physical time semantics.
  2. Constructed binary missing indicator flag `PREV_FIRST_DUE_IS_NA` to preserve structural missingness signal (40.30% raw missing + replaced placeholders).
  3. Formulated absolute elapsed time in years for valid repayment schedule records: `PREV_FIRST_DUE_YEARS` = `abs(DAYS_FIRST_DUE) / 365.25`.
  4. Derived operational payment grace period offset: `PREV_DAYS_DRAWING_TO_DUE` = `DAYS_FIRST_DUE - DAYS_FIRST_DRAWING`.

- **Rationale:**
  - The value `365243` dominates the raw distribution (`Mode = 365,243`) as a database placeholder for non-disbursed, canceled, or unapproved applications. Replacing it with `np.nan` prevents severe tree split distortion.
  - Converting relative negative days to positive years (`PREV_FIRST_DUE_YEARS`) provides clean temporal distance metrics for downstream xAI analysis.
  - Computing `PREV_DAYS_DRAWING_TO_DUE` captures the grace period length between loan disbursement and the first required installment payment.

In [16]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_LAST_DUE_1ST_VERSION ---

# 1. Replace placeholder value 365243 with NaN to resolve systematic data anomaly
df_prev['DAYS_LAST_DUE_1ST_VERSION'] = df_prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan)

# 2. Create binary indicator flag for missing/un-scheduled original maturity dates
df_prev['PREV_LAST_DUE_1ST_VERSION_IS_NA'] = df_prev['DAYS_LAST_DUE_1ST_VERSION'].isna().astype(int)

# 3. Convert valid negative relative days into positive elapsed years for model interpretability
df_prev['PREV_LAST_DUE_1ST_VERSION_YEARS'] = df_prev['DAYS_LAST_DUE_1ST_VERSION'].abs() / 365.25

# 4. Calculate planned contract duration in days (Original Last Due - First Due)
if 'DAYS_FIRST_DUE' in df_prev.columns:
    df_prev['PREV_PLANNED_CONTRACT_DURATION'] = df_prev['DAYS_LAST_DUE_1ST_VERSION'] - df_prev['DAYS_FIRST_DUE']

print("✅ Successfully processed DAYS_LAST_DUE_1ST_VERSION!")
print(
    f"   • Total missing/un-scheduled records flagged: {df_prev['PREV_LAST_DUE_1ST_VERSION_IS_NA'].sum():,} ({df_prev['PREV_LAST_DUE_1ST_VERSION_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Valid original maturity records range: {df_prev['DAYS_LAST_DUE_1ST_VERSION'].min()} to {df_prev['DAYS_LAST_DUE_1ST_VERSION'].max()} days")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_LAST_DUE_1ST_VERSION!
   • Total missing/un-scheduled records flagged: 766,929 (45.92%)
   • Valid original maturity records range: -2801.0 to 2389.0 days
📊 Current dataset shape: 1,670,214 rows | 57 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_LAST_DUE_1ST_VERSION` (Originally Scheduled Last Due Date)

- **Action Taken:**
  1. Replaced systemic anomaly placeholder `365243` with `np.nan` to restore proper relative time semantics.
  2. Constructed binary missing indicator flag `PREV_LAST_DUE_1ST_VERSION_IS_NA` to retain structural missingness signal (40.30% total missingness rate).
  3. Formulated absolute elapsed time in years for valid maturity dates: `PREV_LAST_DUE_1ST_VERSION_YEARS` = `abs(DAYS_LAST_DUE_1ST_VERSION) / 365.25`.
  4. Derived original contract tenor duration: `PREV_PLANNED_CONTRACT_DURATION` = `DAYS_LAST_DUE_1ST_VERSION - DAYS_FIRST_DUE`.

- **Rationale:**
  - The database placeholder `365243` dominates the raw feature distribution (`Mode = 365,243`), representing unapproved, canceled, or non-disbursed loan applications. Replacing it with `np.nan` prevents severe split distortion in tree-based algorithms.
  - `DAYS_LAST_DUE_1ST_VERSION` captures the **originally intended maturity date** agreed upon at loan signing.
  - Computing `PREV_PLANNED_CONTRACT_DURATION` yields the baseline planned repayment timeframe in days, serving as a clean benchmark when compared against actual completion dates.

In [17]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_LAST_DUE ---

# 1. Replace placeholder value 365243 with NaN to resolve systematic data anomaly
df_prev['DAYS_LAST_DUE'] = df_prev['DAYS_LAST_DUE'].replace(365243, np.nan)

# 2. Create binary indicator flag for missing/un-terminated loan maturity dates
df_prev['PREV_LAST_DUE_IS_NA'] = df_prev['DAYS_LAST_DUE'].isna().astype(int)

# 3. Convert valid negative relative days into positive elapsed years for model interpretability
df_prev['PREV_LAST_DUE_YEARS'] = df_prev['DAYS_LAST_DUE'].abs() / 365.25

# 4. Calculate actual contract completion duration (Actual Last Due - First Due)
if 'DAYS_FIRST_DUE' in df_prev.columns:
    df_prev['PREV_ACTUAL_CONTRACT_DURATION'] = df_prev['DAYS_LAST_DUE'] - df_prev['DAYS_FIRST_DUE']

# 5. Formulate maturity deviation/early-settlement offset (Actual Last Due - Planned 1st Version Last Due)
if 'DAYS_LAST_DUE_1ST_VERSION' in df_prev.columns:
    df_prev['PREV_DAYS_EARLY_TERMINATION'] = df_prev['DAYS_LAST_DUE_1ST_VERSION'] - df_prev['DAYS_LAST_DUE']

print("✅ Successfully processed DAYS_LAST_DUE!")
print(
    f"   • Total missing/un-terminated records flagged: {df_prev['PREV_LAST_DUE_IS_NA'].sum():,} ({df_prev['PREV_LAST_DUE_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Valid actual maturity records range: {df_prev['DAYS_LAST_DUE'].min()} to {df_prev['DAYS_LAST_DUE'].max()} days")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_LAST_DUE!
   • Total missing/un-terminated records flagged: 884,286 (52.94%)
   • Valid actual maturity records range: -2889.0 to -2.0 days
📊 Current dataset shape: 1,670,214 rows | 61 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_LAST_DUE` (Actual Last Due Date / Termination Date)

- **Action Taken:**
  1. Replaced systemic anomaly placeholder `365243` with `np.nan` to restore proper relative temporal metrics.
  2. Constructed binary missing indicator flag `PREV_LAST_DUE_IS_NA` to preserve structural missingness signal (40.30% total missingness rate).
  3. Formulated absolute elapsed time in years for valid maturity dates: `PREV_LAST_DUE_YEARS` = `abs(DAYS_LAST_DUE) / 365.25`.
  4. Derived actual completed loan tenor in days: `PREV_ACTUAL_CONTRACT_DURATION` = `DAYS_LAST_DUE - DAYS_FIRST_DUE`.
  5. Computed early settlement / term variance metric: `PREV_DAYS_EARLY_TERMINATION` = `DAYS_LAST_DUE_1ST_VERSION - DAYS_LAST_DUE`.

- **Rationale:**
  - The database placeholder `365243` dominates raw statistics (`Mode = 365,243`), representing active loans, canceled applications, or non-disbursed offers. Replacing it with `np.nan` prevents severe split distortion in tree-based algorithms.
  - `DAYS_LAST_DUE` represents the **actual date of final payment / contract termination**.
  - Comparing actual termination date (`DAYS_LAST_DUE`) against original planned maturity (`DAYS_LAST_DUE_1ST_VERSION`) via `PREV_DAYS_EARLY_TERMINATION` creates a strong behavioral signal, isolating borrowers who consistently settle debt early versus those who extend repayment.

In [18]:
# --- CELL: PREPROCESSING & FEATURE ENGINEERING - DAYS_TERMINATION ---

# 1. Replace placeholder value 365243 with NaN to resolve systematic data anomaly
df_prev['DAYS_TERMINATION'] = df_prev['DAYS_TERMINATION'].replace(365243, np.nan)

# 2. Create binary indicator flag for missing/unterminated loan records
df_prev['PREV_TERMINATION_IS_NA'] = df_prev['DAYS_TERMINATION'].isna().astype(int)

# 3. Convert valid negative relative days into positive elapsed years for model interpretability
df_prev['PREV_TERMINATION_YEARS'] = df_prev['DAYS_TERMINATION'].abs() / 365.25

# 4. Formulate operational offset between actual last due date and formal contract termination
if 'DAYS_LAST_DUE' in df_prev.columns:
    df_prev['PREV_DAYS_DUE_TO_TERMINATION'] = df_prev['DAYS_TERMINATION'] - df_prev['DAYS_LAST_DUE']

print("✅ Successfully processed DAYS_TERMINATION!")
print(
    f"   • Total missing/unterminated records flagged: {df_prev['PREV_TERMINATION_IS_NA'].sum():,} ({df_prev['PREV_TERMINATION_IS_NA'].mean() * 100:.2f}%)")
print(
    f"   • Valid formal termination records range: {df_prev['DAYS_TERMINATION'].min()} to {df_prev['DAYS_TERMINATION'].max()} days")
print(f"📊 Current dataset shape: {df_prev.shape[0]:,} rows | {df_prev.shape[1]} columns")

✅ Successfully processed DAYS_TERMINATION!
   • Total missing/unterminated records flagged: 898,978 (53.82%)
   • Valid formal termination records range: -2874.0 to -2.0 days
📊 Current dataset shape: 1,670,214 rows | 64 columns


### 📌 Preprocessing & Feature Engineering: `DAYS_TERMINATION` (Relative Days to Formal Contract Termination)

- **Action Taken:**
  1. Replaced systemic anomaly placeholder `365243` with `np.nan` to restore proper relative time semantics.
  2. Constructed binary missing indicator flag `PREV_TERMINATION_IS_NA` to preserve structural missingness signal (40.30% total missingness rate).
  3. Formulated absolute elapsed time in years for valid formal contract terminations: `PREV_TERMINATION_YEARS` = `abs(DAYS_TERMINATION) / 365.25`.
  4. Derived operational contract closing lag offset: `PREV_DAYS_DUE_TO_TERMINATION` = `DAYS_TERMINATION - DAYS_LAST_DUE`.

- **Rationale:**
  - The database placeholder `365243` dominates raw statistics (`Mode = 365,243`), representing active loans, canceled applications, or non-disbursed offers. Replacing it with `np.nan` prevents severe split distortion in tree-based algorithms.
  - `DAYS_TERMINATION` represents the **formal administrative closing date** of the loan contract in Home Credit's core banking database.
  - Calculating `PREV_DAYS_DUE_TO_TERMINATION` measures administrative latency between final payment (`DAYS_LAST_DUE`) and formal account closure, isolating clean write-offs from delayed paperwork processing.

In [20]:
# --- SINGLE CELL: STRICT CLEANING, ENCODING, SANITIZATION & EXPORT (V2) ---

import os
import gc
import re
import pandas as pd

print("🚀 Bắt đầu One-Hot Encoding (ép int), Chuẩn hóa tên cột & Làm phẳng (V2)...")

# 1. One-Hot Encoding ép kiểu dtype=int (TRÁNH TẠO KIỂU BOOL GÂY LỖI XGBOOST)
cat_cols = df_prev.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'NAME_CONTRACT_STATUS']

df_prev = pd.get_dummies(df_prev, columns=cat_cols, dummy_na=True, dtype=int)
df_prev.drop(columns=['NAME_CONTRACT_STATUS'], inplace=True, errors='ignore')

# 2. Ép toàn bộ các cột bool/float dạng cờ còn sót lại về kiểu int8/int64
for col in df_prev.select_dtypes(include=['bool']).columns:
    df_prev[col] = df_prev[col].astype(int)

# 3. Gán tiền tố 'PREV_' một lần duy nhất cho toàn bộ cột (trừ khóa chính)
exclude_keys = ['SK_ID_CURR', 'SK_ID_PREV']
rename_dict = {col: f'PREV_{col}' for col in df_prev.columns if col not in exclude_keys and not col.startswith('PREV_')}
df_prev.rename(columns=rename_dict, inplace=True)

# 4. Tạo từ điển phép toán Groupby (Tích hợp luôn đếm SK_ID_PREV)
num_aggregations = {'SK_ID_PREV': 'count'}

for col in df_prev.columns:
    if col in exclude_keys:
        continue
    # Nếu là cờ nhị phân (0/1): Lấy MEAN (tỷ lệ) và SUM (tổng lần)
    if df_prev[col].nunique() == 2 and set(df_prev[col].dropna().unique()).issubset({0, 1}):
        num_aggregations[col] = ['mean', 'sum']
    else:
        # Biến số liên tục: Lấy MIN, MAX, MEAN, SUM, VAR
        num_aggregations[col] = ['min', 'max', 'mean', 'sum', 'var']

prev_agg = df_prev.groupby('SK_ID_CURR').agg(num_aggregations)

# Đổi tên cột đa tầng & Đổi tên cột đếm thành PREV_COUNT
prev_agg.columns = pd.Index(
    [f"{col}_{stat.upper()}" if col != 'SK_ID_PREV' else 'PREV_COUNT' for col, stat in prev_agg.columns])

# 5. CHỐNG LỖI JSON LIGHTGBM: Làm sạch dứt điểm ký tự đặc biệt trong tên cột
prev_agg.columns = [re.sub(r'[^\w_]', '_', col) for col in prev_agg.columns]
prev_agg.columns = [re.sub(r'_{2,}', '_', col) for col in prev_agg.columns]

# 6. Chống phân mảnh bộ nhớ & Lưu file Parquet sạch 100% (ĐƯỜNG DẪN TUYỆT ĐỐI CHUẨN)
prev_agg = prev_agg.copy()
output_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/table/prev_app_clean_fe_v2.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

prev_agg.to_parquet(output_path, compression='snappy')

print("=" * 70)
print(f"🎉 ĐÃ CLEAN TỪ GỐC & LƯU PARQUET V2: {output_path}")
print(f"📊 Kích thước dữ liệu : {prev_agg.shape[0]:,} khách hàng | {prev_agg.shape[1]} thuộc tính")
print("=" * 70)

gc.collect()

🚀 Bắt đầu One-Hot Encoding (ép int), Chuẩn hóa tên cột & Làm phẳng (V2)...
🎉 ĐÃ CLEAN TỪ GỐC & LƯU PARQUET V2: /Users/nguyenminhtri/FinalYearPro/data/processed/table/prev_app_clean_fe_v2.parquet
📊 Kích thước dữ liệu : 338,857 khách hàng | 245 thuộc tính


0